# Civil War News — Notebook 4: Topic Explorer

Loads a saved BERTopic model and runs exploratory analysis.
Runs on your **local machine** — reads from local disk or HuggingFace.

**RAM notes:**
- Model + topic_embeddings_: ~50 MB — always fine
- `embeddings_macberth.npy` (12 GB): only needed for true topic centroids; loaded with `mmap_mode='r'` to avoid RAM spike (reads from disk on demand, ~20 min but never loads fully into RAM)
- `topic_embeddings_macberth.npy` (~13 MB): cached after first computation so centroid rebuild only runs once
- For `visualize_documents()` (not included by default): requires both embeddings + topics (~12 GB + 34 MB)

**Files (local `LOCAL_BASE` or auto-downloaded from HF):**

| File | Size | Source |
|------|------|--------|
| `bertopic_model_macberth/` | 28 MB | local or `civil-war-news` HF |
| `topics_macberth.npy` | 34 MB | local or `civil-war-news` HF |
| `topic_embeddings_macberth.npy` | 13 MB | computed + cached locally |
| `embeddings_macberth.npy` | 12 GB | local only (mmap) |

**Contents:**
- Section 1: Packages & Imports
- Section 2: Configuration
- Section 3: Load Model
- Section 4: Topic Exploration
- Section 5: BERTopic Built-in Visualizations
- Section 6: Custom — Keyword Group Affinity

In [1]:
%pip install -q bertopic sentence-transformers huggingface_hub plotly scipy nbformat
print('Done.')

Note: you may need to restart the kernel to use updated packages.
Done.


In [2]:
import os
import zipfile
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sentence_transformers import models as ST_models
from huggingface_hub import hf_hub_download, login

# Optional HF login — needed only if downloading private repos
try:
    _token = os.environ.get('HF_TOKEN')
    if _token:
        login(token=_token, add_to_git_credential=False)
        print('Logged in to HuggingFace.')
except Exception:
    pass

print('Imports OK.')

Imports OK.


## Section 2: Configuration

In [3]:
# ── Edit these ────────────────────────────────────────────────────────────────
MODEL_NAME = 'macberth'   # 'macberth'  or  'bert_1850_1875'
LOCAL_BASE = r'C:\Users\Patrick\Documents\Projects\Dissertation\Civil-War-News\data\BERTopic'

HF_REPO = 'patrickjcrawford/civil-war-news'   # model, topics.npy, results
# ─────────────────────────────────────────────────────────────────────────────

_MODEL_CONFIGS = {
    'macberth': {
        'model_id': 'emanjavacas/MacBERTh',
        'needs_manual_pooling': True,
    },
    'bert_1850_1875': {
        'model_id': 'Livingwithmachines/bert_1850_1875',
        'needs_manual_pooling': True,
    },
}
CFG      = _MODEL_CONFIGS[MODEL_NAME]
MODEL_ID = CFG['model_id']

print(f'Model : {MODEL_NAME}')
print(f'ID    : {MODEL_ID}')
print(f'Path  : {LOCAL_BASE}')

Model : macberth
ID    : emanjavacas/MacBERTh
Path  : C:\Users\Patrick\Documents\Projects\Dissertation\Civil-War-News\data\BERTopic


## Section 3: Load Model

Loads the BERTopic model from local disk; downloads from HF if not present.
Rebuilds `topic_embeddings_` (needed for `find_topics` and visualizations) and
caches the result to `topic_embeddings_{model}.npy` so this only runs once.

In [ ]:
def _load_embedding_model(cfg):
    model_id = cfg['model_id']
    if cfg['needs_manual_pooling']:
        word_model    = ST_models.Transformer(model_id, max_seq_length=512)
        pooling_model = ST_models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True,
        )
        return SentenceTransformer(modules=[word_model, pooling_model])
    return SentenceTransformer(model_id)

print(f'Loading embedding model: {MODEL_ID} ...')
_emb_model = _load_embedding_model(CFG)
print('Embedding model ready.')

# ── Load BERTopic model (local → HF subfolder fallback) ──────────────────
_pkl = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}.pkl')
_zip = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}.zip')
_dir = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}')

if os.path.exists(_pkl):
    print(f'Loading pickle: {_pkl}')
    topic_model = BERTopic.load(_pkl, embedding_model=_emb_model)
elif os.path.exists(_zip):
    print(f'Extracting zip: {_zip}')
    with zipfile.ZipFile(_zip, 'r') as _zf:
        _zf.extractall(LOCAL_BASE)
    topic_model = BERTopic.load(_dir, embedding_model=_emb_model)
elif os.path.isdir(_dir):
    print(f'Loading safetensors dir: {_dir}')
    topic_model = BERTopic.load(_dir, embedding_model=_emb_model)
else:
    print(f'Not found locally. Downloading from HF ({HF_REPO}/{MODEL_NAME}/) ...')
    _hf_zip = hf_hub_download(repo_id=HF_REPO,
                              filename=f'{MODEL_NAME}/bertopic_model_{MODEL_NAME}.zip',
                              repo_type='dataset')
    with zipfile.ZipFile(_hf_zip, 'r') as _zf:
        _zf.extractall(LOCAL_BASE)
    topic_model = BERTopic.load(_dir, embedding_model=_emb_model)

_n_topics = len(topic_model.get_topic_info()) - 1
print(f'Model loaded: {MODEL_NAME}  ({_n_topics} topics)')

# ── Restore topic_embeddings_ ─────────────────────────────────────────────
_te_path     = os.path.join(LOCAL_BASE, f'topic_embeddings_{MODEL_NAME}.npy')
_topics_path = os.path.join(LOCAL_BASE, f'topics_{MODEL_NAME}.npy')
_embs_path   = os.path.join(LOCAL_BASE, f'embeddings_{MODEL_NAME}.npy')
_N_DOCS      = 8_486_860

_needs_rebuild = (
    topic_model.topic_embeddings_ is None or
    topic_model.topic_embeddings_.shape[1] == 1
)

if not _needs_rebuild:
    print(f'topic_embeddings_ already valid: {topic_model.topic_embeddings_.shape}')
elif os.path.exists(_te_path):
    _te = np.load(_te_path)
    if _te.shape[0] == _n_topics + 1:
        topic_model.topic_embeddings_ = _te
        print(f'Loaded cached topic_embeddings_: {_te.shape}')
        _needs_rebuild = False
    else:
        print(f'Cached shape {_te.shape} mismatches model — recomputing.')
        os.remove(_te_path)

if _needs_rebuild:
    _topic_arr = None

    if os.path.exists(_topics_path):
        _arr = np.load(_topics_path)
        if len(_arr) == _N_DOCS:
            _topic_arr = _arr
        else:
            print(f'topics.npy has {len(_arr):,} entries (expected {_N_DOCS:,}) — trying HF.')

    if _topic_arr is None:
        try:
            print(f'Downloading {MODEL_NAME}/topics_{MODEL_NAME}.npy from HF ...')
            _hf = hf_hub_download(repo_id=HF_REPO,
                                  filename=f'{MODEL_NAME}/topics_{MODEL_NAME}.npy',
                                  repo_type='dataset')
            _arr = np.load(_hf)
            if len(_arr) == _N_DOCS:
                _topic_arr = _arr
                np.save(_topics_path, _topic_arr)
                print(f'Downloaded ({len(_topic_arr):,} entries) and saved locally.')
            else:
                print(f'HF topics.npy has {len(_arr):,} entries — skipping.')
        except Exception as _e:
            print(f'HF download failed: {_e}')

    if _topic_arr is not None and os.path.exists(_embs_path):
        print('Computing true topic centroids via mmap (~10-20 min) ...')
        _doc_embs  = np.load(_embs_path, mmap_mode='r')
        _t_ids     = sorted(set(_topic_arr.tolist()) - {-1})
        _centroids = np.stack([_doc_embs[_topic_arr == t].mean(axis=0) for t in _t_ids])
        topic_model.topic_embeddings_ = np.vstack([
            np.zeros((1, _centroids.shape[1])), _centroids
        ])
        np.save(_te_path, topic_model.topic_embeddings_)
        print(f'topic_embeddings_: {topic_model.topic_embeddings_.shape}  (true centroids, cached)')
    else:
        if _topic_arr is None:
            print('topics.npy unavailable — using keyword approximation.')
        else:
            print('embeddings.npy not local — using keyword approximation.')
        _t_info  = topic_model.get_topic_info()
        _t_ids   = sorted(_t_info.loc[_t_info.Topic != -1, 'Topic'].tolist())
        _t_texts = [' '.join(w for w, _ in topic_model.get_topic(t)[:15]) for t in _t_ids]
        print(f'Encoding {len(_t_ids)} topic keyword strings ...')
        _kw_embs = _emb_model.encode(_t_texts, show_progress_bar=True, normalize_embeddings=True)
        topic_model.topic_embeddings_ = np.vstack([
            np.zeros((1, _kw_embs.shape[1])), _kw_embs
        ])
        np.save(_te_path, topic_model.topic_embeddings_)
        print(f'topic_embeddings_: {topic_model.topic_embeddings_.shape}  (keyword approx, cached)')

print('Ready.')

## Section 4: Topic Exploration

In [5]:
topic_info = topic_model.get_topic_info()
_n_topics  = len(topic_info) - 1
_noise_row = topic_info.loc[topic_info.Topic == -1, 'Count']
_n_noise   = int(_noise_row.values[0]) if len(_noise_row) else 0

print(f'Topics    : {_n_topics}')
print(f'Noise (-1): {_n_noise:,}')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)
topic_info[['Topic', 'Count', 'Name', 'Representation']]

Topics    : 4415
Noise (-1): 1,656,226


,Topic,Count,Name,Representation
0,-1,1656226,-1_gen_tie_says_70,"[gen, tie, says, 70, new, army, city, yesterday, men, war, house, general, al, sale, received, state, president, was..."
1,0,7152,0_ass_ss_bowery_sass,"[ass, ss, bowery, sass, ssh, sad, bowery theatre, psf, tvs, sss, asa, toss, fps, broadway, sea, aka, sassy, ski, sri..."
2,1,6593,1_pay charges_prove property_property pay_years old,"[pay charges, prove property, property pay, years old, inches high, reward, forward prove, charges away, mare, hind,..."
3,2,5950,2_thy_thee_bright_hearts,"[thy, thee, bright, hearts, thou, sweet, love, heart, wave, freedoms, sky, heaven, hath, glory, song, earth, flowers..."
4,3,5832,3_slavery_political_rebellion_rights,"[slavery, political, rebellion, rights, nation, constitution, power, people, policy, party, treason, loyal, struggle..."
...,...,...,...,...
4411,4410,21,4410_equitable_brady street_states office_street post,"[equitable, brady street, states office, street post, assurance, ross, brady, office office, agents, life, 35, ths, ..."
4412,4411,28,4411_nearly double_75 cents_large size_small size,"[nearly double, 75 cents, large size, small size, doz, cents bottle, 90 large, 75, size, 81 bottle, 90 bottle, bottl..."
4413,4412,26,4412_stitch_baker stitch_sold prices_stitch alike,"[stitch, baker stitch, sold prices, stitch alike, alike sides, alike, shuttle stitch, new style, lock stitch, plough..."
4414,4413,17,4413_heartily endorse_endorse_chicago iii_unrivalled,"[heartily endorse, endorse, chicago iii, unrivalled, heartily, scalding, cephalic pills, cephalic, gazette, spading,..."


In [6]:
# Top words for a specific topic — change topic_id as needed.
topic_id = 9
words = topic_model.get_topic(topic_id)
print(f'Topic {topic_id} top words:')
for word, score in words:
    print(f'  {score:.4f}  {word}')

Topic 9 top words:
  0.0041  killed wounded
  0.0040  rebels
  0.0037  loss killed
  0.0035  enemy
  0.0035  killed
  0.0032  force
  0.0032  wounded
  0.0031  cavalry
  0.0029  captured
  0.0029  attacked
  0.0027  prisoners
  0.0027  forces
  0.0024  pieces artillery
  0.0024  loss
  0.0024  skirmish
  0.0023  repulsed
  0.0023  artillery
  0.0022  rebel
  0.0022  rebel force
  0.0022  rebel loss


In [7]:
# Semantic topic search — find topics most similar to a query string.
query = 'slavery emancipation abolition'
similar_topics, similarity = topic_model.find_topics(query, top_n=10)
for t, s in zip(similar_topics, similarity):
    words_str = ', '.join(w for w, _ in topic_model.get_topic(t)[:5])
    print(f'  Topic {t:4d}  sim={s:.3f}  [{words_str}]')

  Topic 3765  sim=0.846  [strong drink, dote, kroner, shannon, zane]
  Topic 1437  sim=0.846  [poverty, disadvantages, demonstrates, ugly, young man]
  Topic 3936  sim=0.844  [anti slavery, ss ss, pf, afr, tss]
  Topic 2529  sim=0.843  [ofhce, sfs, law notary, attorney law, mackerel]
  Topic 2966  sim=0.843  [gage, , , , ]
  Topic 2219  sim=0.842  [cm, kidney complaints, kidney, fewer, bilious]
  Topic 2335  sim=0.841  [aug 23, question negro, special election, act authorizing, suffrage]
  Topic 1489  sim=0.841  [disunion, oaf, unlawful, treaties, wrongs]
  Topic 2549  sim=0.841  [gordon, atlanta, arrived al, cavalry force, forrest]
  Topic 1051  sim=0.841  [augusta georgia, typo, months new, folio, weekly paper]


In [8]:
# Topic representation details for a given topic.
# Note: representative_docs_ (actual article text) is not saved in the safetensors
# format, so get_representative_docs() always returns None after loading from disk.
# What IS available: c-TF-IDF word scores + KeyBERT/MMR aspects from Phase-2.
topic_id = 3

print(f'=== Topic {topic_id}: {topic_model.get_topic_info().loc[topic_model.get_topic_info().Topic == topic_id, "Name"].values[0]} ===\n')

# c-TF-IDF top words (base vocabulary representation)
print('c-TF-IDF top words:')
for word, score in topic_model.get_topic(topic_id):
    print(f'  {score:.4f}  {word}')

# KeyBERT / MMR keyword aspects from Phase-2 update_topics
if hasattr(topic_model, 'topic_aspects_') and topic_model.topic_aspects_:
    for aspect_name, aspect_data in topic_model.topic_aspects_.items():
        if topic_id in aspect_data:
            print(f'\n{aspect_name} keywords:')
            for word, score in aspect_data[topic_id][:10]:
                print(f'  {score:.4f}  {word}')

=== Topic 3: 3_slavery_political_rebellion_rights ===

c-TF-IDF top words:
  0.0017  slavery
  0.0017  political
  0.0016  rebellion
  0.0016  rights
  0.0016  nation
  0.0015  constitution
  0.0015  power
  0.0015  people
  0.0014  policy
  0.0014  party
  0.0014  treason
  0.0014  loyal
  0.0013  struggle
  0.0013  government
  0.0012  civil
  0.0012  peace
  0.0012  civil war
  0.0012  principles
  0.0012  liberty
  0.0012  question

KeyBERT keywords:
  0.8015  american people
  0.7878  patriotism
  0.7803  despotism
  0.7665  civil war
  0.7659  secession
  0.7429  loyal states
  0.7385  people
  0.7370  sacrifices
  0.7360  armies
  0.7357  honor

MMR keywords:
  0.0015  people
  0.0012  civil
  0.0012  civil war
  0.0012  question
  0.0012  institutions
  0.0011  country
  0.0011  freedom
  0.0010  nations
  0.0010  patriotism
  0.0010  history


## Section 5: BERTopic Built-in Visualizations

All cells in this section use plotly and open interactive HTML in the browser.
No large files needed — all viz runs on the loaded model + `topic_embeddings_`.

In [9]:
# Top-word bar chart for selected topics.
# Change `topics` to any list of topic IDs, or set to None for top 8 by count.
topics_to_show = [3, 7, 8, 18, 4, 1, 2, 5]

fig = topic_model.visualize_barchart(
    topics=topics_to_show,
    top_n_topics=8,
    n_words=10,
    title=f'Top Words per Topic — {MODEL_NAME}',
)
fig.show()

In [10]:
# 2D scatter of all topic centroids.
# Topics near each other are semantically similar.
# BERTopic runs a fast internal UMAP on topic_embeddings_ (~4415 points).
fig = topic_model.visualize_topics(
    top_n_topics=None,   # None = all; reduce (e.g. 200) for faster rendering
    title=f'Topic Map — {MODEL_NAME}',
)
fig.show()

In [11]:
# Hierarchical clustering of topics — groups similar topics into a dendrogram.
# Useful for identifying topic families (e.g. all slavery-related topics together).
fig = topic_model.visualize_hierarchy(
    top_n_topics=60,   # reduce for readability; None = all
    title=f'Topic Hierarchy — {MODEL_NAME} (top 60)',
)
fig.show()

In [12]:
# Pairwise topic similarity heatmap for a subset of topics.
# Shows which topics share vocabulary — useful for spotting redundant clusters.
topics_subset = list(range(0, 30))   # first 30 topics

fig = topic_model.visualize_heatmap(
    topics=topics_subset,
    title=f'Topic Similarity Heatmap — {MODEL_NAME}',
)
fig.show()

## Section 6: Custom — Keyword Group Affinity

For each dissertation keyword group, computes cosine similarity between the group centroid
(mean of keyword embeddings) and every topic centroid, returning the most similar topics.
Directly mirrors the `KEYWORD_GROUPS` structure in `Tables_Civil-War-News.R`.

In [13]:
# ── Dissertation keyword groups (mirrors Tables_Civil-War-News.R) ─────────
KEYWORD_GROUPS = {
    'politics':          ['election', 'vote', 'ballot', 'congress', 'party', 'candidate',
                          'republican', 'democrat', 'platform', 'partisan'],
    'slavery_direct':    ['slave', 'slavery', 'bondage', 'enslaved', 'master', 'plantation',
                          'bondman', 'unfree'],
    'slavery_ideology':  ['peculiar institution', 'slave power', 'southern rights',
                          'slavocracy', 'southern institution'],
    'abolition':         ['abolition', 'emancipation', 'freedman', 'manumission',
                          'free negro', 'free black', 'free colored'],
    'race':              ['negro', 'colored man', 'mulatto', 'racial', 'miscegenation',
                          'colored race'],
    'black_military':    ['negro soldier', 'colored troops', 'contrabands',
                          'black regiment', 'colored soldier'],
}

def keyword_group_affinity(groups, top_n=8):
    """
    For each keyword group, find the BERTopic topics most semantically similar.
    Uses pre-computed topic_embeddings_ (no re-encoding needed).

    Returns a DataFrame: group | topic_id | similarity | count | top_words
    """
    # topic_embeddings_ row 0 = outlier (-1), rows 1..N = topics 0..N-1
    # We need the mapping from row index to topic ID
    _t_info   = topic_model.get_topic_info()
    _t_ids    = sorted(_t_info.loc[_t_info.Topic != -1, 'Topic'].tolist())
    _te       = topic_model.topic_embeddings_[1:]  # drop outlier row
    _te_norm  = _te / (np.linalg.norm(_te, axis=1, keepdims=True) + 1e-9)

    rows = []
    for group_name, keywords in groups.items():
        kw_embs  = _emb_model.encode(keywords, normalize_embeddings=True)
        centroid = kw_embs.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)

        sims     = _te_norm @ centroid          # (n_topics,)
        top_idx  = np.argsort(sims)[::-1][:top_n]

        for idx in top_idx:
            t     = _t_ids[idx]
            words = ', '.join(w for w, _ in topic_model.get_topic(t)[:6])
            count = int(_t_info.loc[_t_info.Topic == t, 'Count'].values[0])
            rows.append({
                'group':      group_name,
                'topic_id':   t,
                'similarity': round(float(sims[idx]), 4),
                'count':      count,
                'top_words':  words,
            })

    return pd.DataFrame(rows)


affinity_df = keyword_group_affinity(KEYWORD_GROUPS, top_n=8)

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 80)
affinity_df

,group,topic_id,similarity,count,top_words
0,politics,2966,0.9452,37,"gage, , , , ,"
1,politics,2219,0.7996,35,"cm, kidney complaints, kidney, fewer, bilious, regulator"
2,politics,30,0.7959,1128,"dams, les, pour, midi, par, que"
3,politics,2529,0.7911,35,"ofhce, sfs, law notary, attorney law, mackerel, howell"
4,politics,1883,0.7904,46,"dams, les, par, borne, savior, terry"
5,politics,3336,0.7886,28,"paw, kroner, june ii, antidote, goodie, zane"
6,politics,1489,0.7879,44,"disunion, oaf, unlawful, treaties, wrongs, recognizing"
7,politics,325,0.7860,130,"czapkay, malice, assaults, practitioners, envy, lofty"
8,slavery_direct,2966,0.9541,37,"gage, , , , ,"
9,slavery_direct,30,0.8266,1128,"dams, les, pour, midi, par, que"


In [14]:
# Bar chart of top-N most similar topics per keyword group
import plotly.express as px

_plot_df = affinity_df.copy()
_plot_df['label'] = _plot_df['topic_id'].astype(str) + ': ' + _plot_df['top_words'].str[:40]

fig = px.bar(
    _plot_df,
    x='similarity',
    y='label',
    color='group',
    facet_col='group',
    facet_col_wrap=3,
    orientation='h',
    title='Topic Affinity by Keyword Group',
    labels={'similarity': 'Cosine Similarity', 'label': ''},
    height=900,
)
fig.update_layout(showlegend=False)
fig.update_yaxes(matches=None)
fig.show()